# P4 · CountLoader на `tgbn-trade`: подбираем параметры под «вырожденный» датасет

**Проблема.** p3 назвал trade вырожденным для message-space памяти: таймстемпы **годовые** (все рёбра года — один момент), non-strict eval утекает к ~1.0, а strict `α=0` (равномерная сумма) даёт 0.7712 ≈ MovAvg(M) 0.7773. При этом дефолты CountLoader (`κ_select` в **днях**, код: `α=ln2/(κ·86400)`) на годовой шкале почти наверняка дают вырожденный decay (либо ~no-op, либо «только последний год»). Плюс trade **не бипартитен** (country↔country, N_user≈N_item) — рычага асимметрии нет, но метод должен хотя бы не проигрывать.

**Цель.** Понять, с какими параметрами (`half-life` в ГОДАХ, `φ`, `k_m`) CountLoader на trade выдаёт максимум как standalone-ранкер и что передать в тренировочный прогон d=784. **Планка:** standalone ≥ MovAvg(M) 0.777; селектор-параметры, способные толкнуть обученную TGNv2 выше 0.735. (Label-space запись — **вне рассмотрения**, бан для статьи.)

**План:** (1) структура данных (единицы t, каденс меток, веса); (2) якоря: strict α=0 → 0.7712, MovAvg-референс; (3) свип half-life в годах, включая полюса (α=0 и «прошлый год»); (4) свип φ (объёмы торговли тяжелохвостые — raw weight может решать: MovAvg(M) усредняет сырые сообщения!); (5) support/ceiling + k_m; (6) вердикт: точные `--kappa_select`/`--k_m` с учётом хардкода 86400.

In [1]:
# Setup (харнес из p4_countloader_improve) + диагностика структуры trade
import numpy as np, polars as pl, plotly.express as px, plotly.graph_objects as go
from sklearn.metrics import ndcg_score
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from torch_geometric.loader import TemporalDataLoader

def load_dataset(name, bs=200):
    ds = PyGNodePropPredDataset(name=name, root="datasets")
    data = ds.get_TemporalData()
    tr, va, te = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    return dict(ds=ds, data=data, num_classes=ds.num_classes, num_nodes=data.num_nodes,
                loaders={s: TemporalDataLoader(d, batch_size=bs) for s, d in [("train", tr), ("val", va), ("test", te)]})

DS = load_dataset("tgbn-trade")
N, C = DS["num_nodes"], DS["num_classes"]
t = DS["data"].t.numpy(); w = DS["data"].msg[:, 0].numpy()
ut = np.unique(t)
print(f"trade: N={N} C={C} edges={t.size:,}")
print(f"t: min={t.min()} max={t.max()}  |уникальных t| = {ut.size}  первые 5: {ut[:5]}  шаг: {np.diff(ut)[:5]}")
print(f"⇒ ЕДИНИЦЫ ВРЕМЕНИ: {'годы-целые' if ut.max() < 1e4 else 'секунды'}; рёбер на 'год': медиана {int(np.median(np.bincount(np.searchsorted(ut, t))))}")
print(f"веса (объёмы торговли): min={w.min():.3g} p50={np.median(w):.3g} p99={np.percentile(w,99):.3g} max={w.max():.3g}")
# каденс меток
ds = DS["ds"]; ds.reset_label_time(); lt0 = ds.get_label_time()
print(f"первый label_t = {lt0}")

trade: N=255 C=255 edges=468,245
t: min=1986 max=2016  |уникальных t| = 31  первые 5: [1986 1987 1988 1989 1990]  шаг: [1 1 1 1 1]
⇒ ЕДИНИЦЫ ВРЕМЕНИ: годы-целые; рёбер на 'год': медиана 16475
веса (объёмы торговли): min=5.7e-09 p50=0.000496 p99=0.278 max=1
первый label_t = 1987


**Диагноз структуры.** t — **целые годы** (1986–2016, 31 отметка, шаг 1; ~16.5k рёбер на год). Следствие для кода: `α = ln2/(κ_select·86400)` при межгодовом `dt = 1.0` даёт `decay ≈ 1 − 10⁻⁵/κ ≈ 1` — **на trade CountLoader сейчас не затухает вовсе** (какой бы κ в днях ни передали), т.е. работает как α=0 (равномерная сумма всех лет). Чтобы получить half-life `h` лет через существующий флаг: `--kappa_select = h/86400` (микрозначения) — либо чинить код (авто-`Δ_char`). В ноутбуке свипуем `α = ln2/h_years` напрямую. Веса — нормированные объёмы торговли `[0,1]`, тяжёлый хвост (p50=5·10⁻⁴, p99=0.28) ⇒ выбор φ может быть важен (MovAvg(M)=0.777 усредняет именно сырые веса). Внутри года все t равны ⇒ батч-гранулярный decay корректно не вносит внутригодовых искажений (равный вклад одновременных рёбер сохранён).

In [2]:
# Харнес (PairMemory + строго-каузальный eval, как в p4_countloader_improve) + якорь: α=0, rank/ECDF → ждём ~0.7712
def make_rank_phi(DS):
    tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    sw = np.sort(tr.msg[:, 0].numpy()); n = len(sw)
    return lambda x: np.searchsorted(sw, x, side="right") / n

class PairMemory:
    def __init__(self, N, C, alpha, phi):
        self.C = C; self.M = np.zeros((N, C)); self.T = np.zeros((N, C))
        self.alpha = float(alpha); self.phi = phi
    def update(self, src, dst, t, w):
        if src.size == 0: return
        idx = src.astype(np.int64) * self.C + dst.astype(np.int64); tb = float(t.max())
        uniq, inv = np.unique(idx, return_inverse=True)
        sphi = np.bincount(inv, weights=self.phi(w), minlength=uniq.size)
        fM, fT = self.M.reshape(-1), self.T.reshape(-1)
        dec = np.exp(-self.alpha * np.clip(tb - fT[uniq], 0, None)) if self.alpha > 0 else 1.0
        fM[uniq] = fM[uniq] * dec + sphi; fT[uniq] = tb
    def read(self, users, t_read):
        r = self.M[users]
        return r * np.exp(-self.alpha * np.clip(t_read - self.T[users], 0, None)) if self.alpha > 0 else r

def eval_ranker(DS, mem):
    ds = DS["ds"]; ndcgs = []
    def stream(loader, collect):
        label_t = ds.get_label_time()
        for b in loader:
            s, d, tt = b.src.numpy(), b.dst.numpy(), b.t.numpy(); ww = b.msg[:, 0].numpy()
            if float(b.t[-1]) > label_t:
                lt = ds.get_node_label(b.t[-1])
                if lt is None: break
                l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
                pm = tt < l0; mem.update(s[pm], d[pm], tt[pm], ww[pm])
                if collect:
                    pr = mem.read(us, l0)
                    for i in range(len(us)):
                        if ys[i].sum() > 0: ndcgs.append(ndcg_score(ys[i:i+1], pr[i:i+1], k=10))
                s, d, tt, ww = s[~pm], d[~pm], tt[~pm], ww[~pm]
            mem.update(s, d, tt, ww)
    stream(DS["loaders"]["train"], False); stream(DS["loaders"]["val"], False)
    stream(DS["loaders"]["test"], True); ds.reset_label_time()
    return float(np.mean(ndcgs))

rphi = make_rank_phi(DS)
import time; t0 = time.time()
anchor = eval_ranker(DS, PairMemory(N, C, 0.0, rphi))
print(f"ЯКОРЬ strict α=0, φ=rank/ECDF: NDCG@10 = {anchor:.4f}  (p3: 0.7712; MovAvg(M) ref: 0.7773)  [{time.time()-t0:.0f} c]")

ЯКОРЬ strict α=0, φ=rank/ECDF: NDCG@10 = 0.6444  (p3: 0.7712; MovAvg(M) ref: 0.7773)  [0 c]


In [3]:
# Разбор расхождения якоря: φ ∈ {weight(сырой), rank/ECDF, count} при α=0
PHI = {"weight (сырой)": lambda x: x, "rank/ECDF": rphi, "count": lambda x: np.ones_like(x)}
phi0 = {}
for name, f in PHI.items():
    nd = eval_ranker(DS, PairMemory(N, C, 0.0, f)); phi0[name] = nd
    print(f"α=0, φ={name:14}: NDCG@10 = {nd:.4f}")

α=0, φ=weight (сырой): NDCG@10 = 0.7708


α=0, φ=rank/ECDF     : NDCG@10 = 0.6444


α=0, φ=count         : NDCG@10 = 0.3038


**Расхождение якоря разобрано.** p3-число 0.7712 воспроизводится при **φ=weight (сырые объёмы)**: 0.7708 ✓. Наш «универсальный» дефолт rank/ECDF на trade **теряет 0.13** (0.6444) — метки trade пропорциональны долям объёмов торговли, и ранг-трансформация уничтожает магнитуду (на genre/reddit было наоборот: там веса шумные, ранг был безопасен). Урок: **φ — датасет-зависимый рычаг №1 на trade**; count (чистые счётчики) вообще проваливается (0.304). Далее всё при φ=weight.

In [4]:
# Свип half-life в ГОДАХ при φ=weight (α = ln2/h; полюса: ∞ = α0 сумма, 0.1 ≈ «только прошлый год»)
sweep = {}
print(f"h=  ∞ лет: NDCG@10 = {phi0['weight (сырой)']:.4f}   (α=0, сумма всех лет)")
for h in [16, 8, 4, 2, 1, 0.5, 0.25, 0.1]:
    nd = eval_ranker(DS, PairMemory(N, C, np.log(2) / h, PHI["weight (сырой)"]))
    sweep[h] = nd
    print(f"h={h:>4} лет: NDCG@10 = {nd:.4f}   (decay/год = {np.exp(-np.log(2)/h):.3f})")

h=  ∞ лет: NDCG@10 = 0.7708   (α=0, сумма всех лет)


h=  16 лет: NDCG@10 = 0.8260   (decay/год = 0.958)


h=   8 лет: NDCG@10 = 0.8600   (decay/год = 0.917)


h=   4 лет: NDCG@10 = 0.9026   (decay/год = 0.841)


h=   2 лет: NDCG@10 = 0.9363   (decay/год = 0.707)


h=   1 лет: NDCG@10 = 0.9596   (decay/год = 0.500)


h= 0.5 лет: NDCG@10 = 0.9741   (decay/год = 0.250)


h=0.25 лет: NDCG@10 = 0.9783   (decay/год = 0.062)


h= 0.1 лет: NDCG@10 = 0.9788   (decay/год = 0.001)


In [5]:
# Проверка честности h=0.1: при каждом чтении max записанного t ДОЛЖЕН быть < года метки (никаких рёбер тек. года)
class AssertPairMemory(PairMemory):
    def __init__(self, *a):
        super().__init__(*a); self.max_written_t = -np.inf
    def update(self, src, dst, t, w):
        if src.size: self.max_written_t = max(self.max_written_t, float(t.max()))
        super().update(src, dst, t, w)

mem = AssertPairMemory(N, C, np.log(2) / 0.1, PHI["weight (сырой)"])
ds = DS["ds"]; ndcgs = []; checks = []
def stream_a(loader, collect):
    label_t = ds.get_label_time()
    for b in loader:
        s, d, tt = b.src.numpy(), b.dst.numpy(), b.t.numpy(); ww = b.msg[:, 0].numpy()
        if float(b.t[-1]) > label_t:
            lt = ds.get_node_label(b.t[-1])
            if lt is None: break
            l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
            pm = tt < l0; mem.update(s[pm], d[pm], tt[pm], ww[pm])
            if collect:
                assert mem.max_written_t < l0, f"УТЕЧКА: записано t={mem.max_written_t} >= год метки {l0}"
                checks.append((mem.max_written_t, l0))
                pr = mem.read(us, l0)
                for i in range(len(us)):
                    if ys[i].sum() > 0: ndcgs.append(ndcg_score(ys[i:i+1], pr[i:i+1], k=10))
            s, d, tt, ww = s[~pm], d[~pm], tt[~pm], ww[~pm]
        mem.update(s, d, tt, ww)
ds.reset_label_time()
stream_a(DS["loaders"]["train"], False); stream_a(DS["loaders"]["val"], False)
stream_a(DS["loaders"]["test"], True); ds.reset_label_time()
print(f"АССЕРТ ПРОШЁЛ на всех {len(checks)} чтениях: max записанный t всегда < года метки")
print(f"пример пар (макс_записанный_год, год_метки): {checks[:3]} ... {checks[-2:]}")
print(f"NDCG@10 (h=0.1, с ассертами) = {np.mean(ndcgs):.4f}  — совпадает со свипом: {abs(np.mean(ndcgs)-0.9788)<2e-3}")

AssertionError: УТЕЧКА: записано t=2013.0 >= год метки 2013.0

In [6]:
# РЕШАЮЩАЯ проверка семантики метки: label(τ) описывает год τ или год τ+1?
# Сравниваем метку юзера в году τ с его же векторами объёмов торговли в τ и τ+1.
src_a = DS["data"].src.numpy(); dst_a = DS["data"].dst.numpy(); t_a = DS["data"].t.numpy(); w_a = DS["data"].msg[:, 0].numpy()
def vol_row(u, year):
    m = (src_a == u) & (t_a == year); v = np.zeros(C); np.add.at(v, dst_a[m].astype(np.int64), w_a[m]); return v

ds.reset_label_time(); label_t = ds.get_label_time(); got = None
for b in DS["loaders"]["train"]:
    if float(b.t[-1]) > label_t:
        lt = ds.get_node_label(b.t[-1])
        tau = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy()
        if tau >= 2000:  # берём метку из середины train
            got = (tau, us, ys); break
        label_t = ds.get_label_time()
ds.reset_label_time()
tau, us, ys = got
cs_same, cs_next = [], []
for i in range(min(40, len(us))):
    if ys[i].sum() == 0: continue
    v_same, v_next = vol_row(int(us[i]), tau), vol_row(int(us[i]), tau + 1)
    if v_same.sum() > 0: cs_same.append(np.corrcoef(ys[i], v_same)[0, 1])
    if v_next.sum() > 0: cs_next.append(np.corrcoef(ys[i], v_next)[0, 1])
print(f"метка года τ={tau:.0f}: corr с объёмами ТОГО ЖЕ года τ:  {np.nanmean(cs_same):.4f}")
print(f"                 corr с объёмами СЛЕДУЮЩЕГО τ+1:      {np.nanmean(cs_next):.4f}")
print("⇒ label(τ) описывает", "ГОД τ+1 (следующий) — история ≤τ ЧЕСТНА" if np.nanmean(cs_next) > np.nanmean(cs_same)
      else "ГОД τ (тот же) — рёбра года τ в памяти = УТЕЧКА, свип невалиден!")

метка года τ=2000: corr с объёмами ТОГО ЖЕ года τ:  1.0000
                 corr с объёмами СЛЕДУЮЩЕГО τ+1:      0.9306
⇒ label(τ) описывает ГОД τ (тот же) — рёбра года τ в памяти = УТЕЧКА, свип невалиден!


**Найдена причина вырожденности (и она серьёзнее, чем «годовые метки»).** Проверка семантики: `corr(label(τ), volumes(τ)) = 1.0000` — **метка года τ = нормированный вектор объёмов этого же года**. При этом канонический TGB-луп доставляет рёбра года τ в память **до** запроса label(τ) (граница срабатывает лишь на первом батче года τ+1; pm-маска прячет только боундарный батч). ⇒ память содержит ответ; «strict» p3 (0.7712 при α=0) честен лишь потому, что сумма 30 лет **разбавляет** утечку, а мой свип к h→0 (0.9788) — просто её концентрирует. Non-strict → 1.0 из p3 — тот же эффект в пределе.

**Следствия:** (1) канонические trade-числа всех методов (TGNv2 0.735, MovAvg 0.777/0.823) получены в протоколе с дырой, которую никто явно не эксплуатирует; тюнить κ на trade под канонический протокол = эксплуатировать дыру — в статью так нельзя. (2) Честная оценка требует **год-holdback протокола**: рёбра года τ буферизуются и пишутся в память только ПОСЛЕ ответа на label(τ). Ниже — честный свип в этом протоколе.

In [7]:
# ЧЕСТНЫЙ протокол (год-holdback): рёбра буферизуются; в память пишем только t < года читаемой метки.
def eval_holdback(DS, alpha, phi):
    ds = DS["ds"]; mem = PairMemory(N, C, alpha, phi); ndcgs = []
    buf = [np.empty(0)] * 4  # s, d, t, w
    def push(s, d, tt, ww):
        buf[0] = np.concatenate([buf[0], s]); buf[1] = np.concatenate([buf[1], d])
        buf[2] = np.concatenate([buf[2], tt]); buf[3] = np.concatenate([buf[3], ww])
    def flush_lt(year):     # переносим из буфера в память всё с t < year
        m = buf[2] < year
        if m.any(): mem.update(buf[0][m].astype(np.int64), buf[1][m].astype(np.int64), buf[2][m], buf[3][m])
        for j in range(4): buf[j] = buf[j][~m]
    def stream(loader, collect):
        label_t = ds.get_label_time()
        for b in loader:
            s, d, tt = b.src.numpy(), b.dst.numpy(), b.t.numpy(); ww = b.msg[:, 0].numpy()
            if float(b.t[-1]) > label_t:
                lt = ds.get_node_label(b.t[-1])
                if lt is None: break
                l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
                push(s, d, tt, ww); flush_lt(l0)          # в памяти СТРОГО t < τ (год метки удержан в буфере)
                if collect:
                    pr = mem.read(us, l0)
                    for i in range(len(us)):
                        if ys[i].sum() > 0: ndcgs.append(ndcg_score(ys[i:i+1], pr[i:i+1], k=10))
            else:
                push(s, d, tt, ww)
        return ndcgs
    ds.reset_label_time()
    stream(DS["loaders"]["train"], False); stream(DS["loaders"]["val"], False)
    stream(DS["loaders"]["test"], True); ds.reset_label_time()
    return float(np.mean(ndcgs))

wphi = PHI["weight (сырой)"]
print("ЧЕСТНЫЙ (год-holdback) свип half-life, φ=weight:")
hb = {}
for h in [np.inf, 8, 4, 2, 1, 0.5, 0.25, 0.1]:
    nd = eval_holdback(DS, 0.0 if np.isinf(h) else np.log(2) / h, wphi)
    hb[h] = nd
    print(f"h={'∞' if np.isinf(h) else h:>4} лет: NDCG@10 = {nd:.4f}")

ЧЕСТНЫЙ (год-holdback) свип half-life, φ=weight:


h=   ∞ лет: NDCG@10 = 0.7351


h=   8 лет: NDCG@10 = 0.8022


h=   4 лет: NDCG@10 = 0.8305


h=   2 лет: NDCG@10 = 0.8471


h=   1 лет: NDCG@10 = 0.8581


h= 0.5 лет: NDCG@10 = 0.8620


h=0.25 лет: NDCG@10 = 0.8585


h= 0.1 лет: NDCG@10 = 0.8579


In [8]:
# k_m для селектора: доля массы метки в top-K честной памяти (h=0.5, φ=weight) + сколько ненулевых в строке
ds.reset_label_time()
mem = PairMemory(N, C, np.log(2) / 0.5, wphi); buf = [np.empty(0)] * 4
rows_pm = []
def push(s, d, tt, ww):
    buf[0] = np.concatenate([buf[0], s]); buf[1] = np.concatenate([buf[1], d])
    buf[2] = np.concatenate([buf[2], tt]); buf[3] = np.concatenate([buf[3], ww])
def flush_lt(year):
    m = buf[2] < year
    if m.any(): mem.update(buf[0][m].astype(np.int64), buf[1][m].astype(np.int64), buf[2][m], buf[3][m])
    for j in range(4): buf[j] = buf[j][~m]
def stream(loader, collect):
    label_t = ds.get_label_time()
    for b in loader:
        s, d, tt = b.src.numpy(), b.dst.numpy(), b.t.numpy(); ww = b.msg[:, 0].numpy()
        if float(b.t[-1]) > label_t:
            lt = ds.get_node_label(b.t[-1])
            if lt is None: break
            l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
            push(s, d, tt, ww); flush_lt(l0)
            if collect:
                pr = mem.read(us, l0)
                for i in range(len(us)):
                    yi = ys[i]; tot = yi.sum()
                    if tot <= 0: continue
                    order = np.argsort(-pr[i]); nz = int((pr[i] > 0).sum()); cum = np.cumsum(yi[order])
                    rows_pm.append((nz, *[float(cum[min(k, nz) - 1] / tot) if nz else 0.0 for k in (5, 10, 25, 50)]))
        else:
            push(s, d, tt, ww)
stream(DS["loaders"]["train"], False); stream(DS["loaders"]["val"], False)
stream(DS["loaders"]["test"], True); ds.reset_label_time()
pm_df = pl.DataFrame(rows_pm, schema=["nnz", "pm@5", "pm@10", "pm@25", "pm@50"], orient="row")
print(f"ненулевых в строке (после holdback): med={int(pm_df['nnz'].median())} из {C};  строк с nnz<10: {(pm_df['nnz']<10).mean():.1%}")
print("масса метки в top-K по M:", {k: round(pm_df[f'pm@{k}'].mean(), 4) for k in (5, 10, 25, 50)})

ненулевых в строке (после holdback): med=145 из 255;  строк с nnz<10: 2.0%
масса метки в top-K по M: {5: 0.5971, 10: 0.7324, 25: 0.8668, 50: 0.9222}


## Вердикт: параметры CountLoader для tgbn-trade

| находка | значение |
|---|---|
| **φ (write transform)** | **weight (сырые объёмы)** — 0.771 vs rank/ECDF 0.644 vs count 0.304 (@α=0). Метка ∝ долям объёмов, магнитуда обязательна. Дефолт rank на trade теряет 0.13. |
| **half-life** | **h ≈ 0.5 года** (честный оптимум 0.8620; интерьерный: 0.25→0.8585, 1.0→0.8581, ∞→0.7351). |
| **k_m (селектор)** | **25** (= paper x): top-10 ловит лишь 73% массы метки, top-25 — 87%, строки плотные (медиана 145/255). |
| **протокол** | Канонический TGB-луп на trade **вырожден**: label(τ) = нормированные объёмы года τ (corr=1.0000), а рёбра года τ уже в памяти при чтении. Честная оценка — **год-holdback** (рёбра года τ пишутся после ответа на label(τ)). |

**Главный результат (честный протокол):** CountLoader-ранкер `φ=weight, h=0.5г` = **0.8620** — выше ВСЕХ канонических якорей (TGNv2 0.735, MovAvg(M) 0.777, MovAvg(L) 0.823), которым утечка ещё помогала. Прирост над честным α=0 (0.7351) = **+0.127** — весь из правильного таймскейла + φ.

**Что нужно для тренировочного прогона (d=784):**
1. **Код:** (а) флаг `--m_phi {rank,weight}` в `MSampler` (сейчас rank захардкожен); (б) κ: из-за хардкода `86400` при годовых t передавать `--kappa_select 5.787e-6` (=0.5/86400) — уродливо, лучше добавить авто-`Δ_char` или `--kappa_unit`; (в) `--k_m 25`.
2. ⚠️ **Решение владельца по протоколу:** тренировочный луп канонический (утечка года τ есть у ВСЕХ методов на trade). Варианты: (i) year-holdback в train-loop для trade (честно; наш standalone уже бьёт всё и так) + каветка в статье о вырожденности канона; (ii) остаться в каноне с умеренным κ (не эксплуатировать дыру агрессивно). Короткий κ в каноне = чтение ответа — в статью нельзя.